# PlanProof Ablation Study Analysis
Evaluating the contribution of each architectural component to planning compliance validation.

In [ ]:
import json
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Try importing visualization deps
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns
    import numpy as np
except ImportError as e:
    print(f"Install eval deps: pip install -e '.[eval]'\nMissing: {e}")

# PlanProof imports
from planproof.evaluation.results import load_all_results
from planproof.evaluation.metrics import (
    compute_confusion_matrix, compute_recall, compute_precision,
    compute_f2_score, compute_automation_rate, bootstrap_ci, cohens_h,
)

# Style
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150
RESULTS_DIR = Path("../data/results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading

In [ ]:
results = load_all_results(RESULTS_DIR)
if not results:
    print("No results found. Run: python scripts/run_ablation.py first")
else:
    print(f"Loaded {len(results)} experiment results")

# Build a DataFrame for analysis
rows = []
for exp in results:
    for rr in exp.rule_results:
        rows.append({
            "config": exp.config_name,
            "set_id": exp.set_id,
            "rule_id": rr.rule_id,
            "ground_truth": rr.ground_truth_outcome,
            "predicted": rr.predicted_outcome,
            "correct": rr.ground_truth_outcome == rr.predicted_outcome,
        })
df = pd.DataFrame(rows)
print(f"Total rule evaluations: {len(df)}")
df.head()

## 2. Configuration Comparison Summary

In [ ]:
CONFIG_ORDER = [
    "naive_baseline", "strong_baseline",
    "ablation_b", "ablation_c", "ablation_d", "full_system"
]
CONFIG_LABELS = {
    "naive_baseline": "Naive LLM",
    "strong_baseline": "Strong LLM (CoT)",
    "ablation_b": "Ablation B\n(No SNKG)",
    "ablation_c": "Ablation C\n(No Gating)",
    "ablation_d": "Ablation D\n(No Assessability)",
    "full_system": "Full System",
}

summary_rows = []
for config in CONFIG_ORDER:
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue
    cm = compute_confusion_matrix(config_results)
    recall = compute_recall(cm)
    precision = compute_precision(cm)
    f2 = compute_f2_score(cm)
    auto_rate = compute_automation_rate(config_results)
    summary_rows.append({
        "Configuration": CONFIG_LABELS.get(config, config),
        "Recall": f"{recall:.2%}",
        "Precision": f"{precision:.2%}",
        "F2 Score": f"{f2:.2%}",
        "Automation Rate": f"{auto_rate:.2%}",
        "NOT_ASSESSABLE": cm.get("not_assessable", 0),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

> **Note:** Recall, Precision, and F2 Score are only meaningful when the evaluation set contains **both compliant (no-violation) and non-compliant (violation) plan cases**. If all test cases are compliant, all violation-based metrics will be 0 and the comparison across configurations is not informative. Ensure the ablation dataset includes a representative mix before drawing conclusions.

## 3. Component Contribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

metrics = ["Recall", "Precision", "F2 Score"]
colors = sns.color_palette("husl", len(CONFIG_ORDER))

any_nonzero = False  # track whether all bars are zero (no compliant/non-compliant mix)

for ax, metric in zip(axes, metrics):
    vals = []
    labels = []
    for i, config in enumerate(CONFIG_ORDER):
        config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
        if not config_results:
            continue
        cm = compute_confusion_matrix(config_results)
        if metric == "Recall":
            v = compute_recall(cm)
        elif metric == "Precision":
            v = compute_precision(cm)
        else:
            v = compute_f2_score(cm)
        vals.append(v)
        labels.append(CONFIG_LABELS.get(config, config))
        if v > 0:
            any_nonzero = True

    bars = ax.bar(range(len(vals)), vals, color=colors[:len(vals)])
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.1)
    ax.set_title(metric, fontweight='bold')
    # Add value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)

if not any_nonzero:
    fig.text(0.5, 0.01,
             "All metrics are 0 — results may contain only compliant (no-violation) cases.\n"
             "Run ablation with a mix of compliant and non-compliant plans for meaningful bars.",
             ha='center', va='bottom', fontsize=9, color='red', style='italic')

plt.suptitle("Ablation Study: Component Contribution to Compliance Validation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ablation_comparison.png", bbox_inches='tight', dpi=300)
plt.show()

## 4. Automation Rate vs Accuracy Trade-off
This plot shows the key innovation: by allowing NOT_ASSESSABLE verdicts, the full system
achieves higher precision on the rules it DOES assess, while honestly flagging gaps.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for i, config in enumerate(CONFIG_ORDER):
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue
    cm = compute_confusion_matrix(config_results)
    recall = compute_recall(cm)
    auto_rate = compute_automation_rate(config_results)
    label = CONFIG_LABELS.get(config, config)
    ax.scatter(auto_rate, recall, s=200, c=[colors[i]], zorder=5, edgecolors='black', linewidths=1)
    ax.annotate(label, (auto_rate, recall), textcoords="offset points",
                xytext=(10, 10), fontsize=9, ha='left')

ax.set_xlabel("Automation Rate (% rules assessable)", fontsize=12)
ax.set_ylabel("Violation Recall", fontsize=12)
ax.set_title("Automation Rate vs Recall Trade-off\nHigher-right is better", fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(-0.05, 1.15)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "automation_vs_recall.png", bbox_inches='tight', dpi=300)
plt.show()

## 5. Per-Rule Performance Heatmap

In [ ]:
rules = sorted(df['rule_id'].unique())
configs_present = [c for c in CONFIG_ORDER if c in df['config'].values]

accuracy_matrix = []
for config in configs_present:
    row = []
    for rule in rules:
        subset = df[(df['config'] == config) & (df['rule_id'] == rule)]
        if len(subset) == 0:
            row.append(float('nan'))
        else:
            row.append(subset['correct'].mean())
    accuracy_matrix.append(row)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(rules)))
ax.set_xticklabels(rules, fontsize=10)
ax.set_yticks(range(len(configs_present)))
ax.set_yticklabels([CONFIG_LABELS.get(c, c) for c in configs_present], fontsize=9)

# Add text annotations
for i in range(len(configs_present)):
    for j in range(len(rules)):
        val = accuracy_matrix[i][j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=10,
                    color='white' if val < 0.5 else 'black')

plt.colorbar(im, label='Accuracy')
ax.set_title("Per-Rule Accuracy by Configuration", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_rule_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

## 6. NOT_ASSESSABLE Verdict Analysis
The assessability engine is the core research contribution. This section analyses when and why
rules are classified as NOT_ASSESSABLE.

In [ ]:
not_assessable = df[df['predicted'] == 'NOT_ASSESSABLE']
if len(not_assessable) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By config
    na_by_config = not_assessable.groupby('config').size()
    na_by_config.plot(kind='barh', ax=axes[0], color=sns.color_palette("husl", len(na_by_config)))
    axes[0].set_xlabel("Count of NOT_ASSESSABLE verdicts")
    axes[0].set_title("NOT_ASSESSABLE by Configuration")

    # By rule
    na_by_rule = not_assessable.groupby('rule_id').size()
    na_by_rule.plot(kind='barh', ax=axes[1], color=sns.color_palette("Set2", len(na_by_rule)))
    axes[1].set_xlabel("Count of NOT_ASSESSABLE verdicts")
    axes[1].set_title("NOT_ASSESSABLE by Rule")

    plt.suptitle("Assessability Analysis", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "not_assessable_analysis.png", bbox_inches='tight', dpi=300)
    plt.show()
else:
    print("No NOT_ASSESSABLE verdicts found in results.")

## 7. Statistical Confidence
Bootstrap confidence intervals for key metrics (1000 resamples).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
y_positions = []
y_labels = []

for i, config in enumerate(CONFIG_ORDER):
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue

    # Bootstrap recall
    recalls = []
    rng = np.random.RandomState(42)
    for _ in range(1000):
        sample = rng.choice(config_results, size=len(config_results), replace=True)
        cm = compute_confusion_matrix(list(sample))
        recalls.append(compute_recall(cm))

    mean = np.mean(recalls)

    # Handle degenerate cases: all recalls identical (no variance) or mean is 0.0
    if mean == 0.0 or len(set(recalls)) == 1:
        lower, upper = mean, mean
    else:
        lower = np.percentile(recalls, 2.5)
        upper = np.percentile(recalls, 97.5)

    # Clamp to avoid negative xerr values (can occur with floating-point edge cases)
    xerr_lo = max(0.0, mean - lower)
    xerr_hi = max(0.0, upper - mean)

    ax.errorbar(mean, i, xerr=[[xerr_lo], [xerr_hi]], fmt='o', markersize=8,
                capsize=5, color=colors[i], linewidth=2)
    y_positions.append(i)
    y_labels.append(CONFIG_LABELS.get(config, config))

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Violation Recall (95% Bootstrap CI)")
ax.set_title("Recall with Confidence Intervals", fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.15)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3, label='50% baseline')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bootstrap_ci.png", bbox_inches='tight', dpi=300)
plt.show()

## 8. Statistical Comparisons
Cohen's h effect sizes: Full System vs each ablation.

In [ ]:
full_results = [r for exp in results for r in exp.rule_results if exp.config_name == "full_system"]
if full_results:
    full_cm = compute_confusion_matrix(full_results)
    full_recall = compute_recall(full_cm)

    effect_sizes = []
    for config in ["naive_baseline", "strong_baseline", "ablation_b", "ablation_c", "ablation_d"]:
        config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
        if not config_results:
            continue
        cm = compute_confusion_matrix(config_results)
        r = compute_recall(cm)
        h = cohens_h(full_recall, r)
        effect_sizes.append({
            "Comparison": f"Full System vs {CONFIG_LABELS.get(config, config)}",
            "Full Recall": f"{full_recall:.2%}",
            "Other Recall": f"{r:.2%}",
            "Cohen's h": f"{h:.3f}",
            "Effect": "Large" if abs(h) > 0.8 else "Medium" if abs(h) > 0.5 else "Small" if abs(h) > 0.2 else "Negligible",
        })
    pd.DataFrame(effect_sizes)

## 9. Qualitative Error Analysis
For each misclassification, document what went wrong and why.

In [ ]:
misclassified = df[~df['correct']]
print(f"Total misclassifications: {len(misclassified)}")
if len(misclassified) > 0:
    for _, row in misclassified.iterrows():
        print(f"\n--- {row['config']} | {row['set_id']} | {row['rule_id']} ---")
        print(f"Ground truth: {row['ground_truth']}, Predicted: {row['predicted']}")
        print("Analysis: [TODO: explain why this happened]")

## 10. Key Findings for Dissertation

### Summary
[Fill in after running experiments]

### Tables for LaTeX export

In [ ]:
# Export summary table for LaTeX
if len(summary_df) > 0:
    print(summary_df.to_latex(index=False, caption="Ablation Study Results", label="tab:ablation"))
    summary_df.to_csv(FIGURES_DIR / "summary_table.csv", index=False)
    print(f"\nFigures saved to {FIGURES_DIR}")